# VECTOR DB

In [15]:
from qdrant_client import QdrantClient
import os
import qdrant_client
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex, StorageContext, Settings, SimpleDirectoryReader, SummaryIndex
from llama_index.core.schema import TextNode
from llama_index.embeddings.openai import OpenAIEmbedding
from dotenv import load_dotenv
from qdrant_client.http.models import (
    VectorParams,
    SparseVectorParams,
    Distance
)
import json
import logging
import time
from pathlib import Path
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    granite_picture_description
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import ImageRefMode, PictureItem 
from docling.chunking import HybridChunker, HierarchicalChunker
from docling.datamodel.document import DoclingDocument
from llama_index.llms.openai_like import OpenAILike
#qua fatto porcata per inserire utils
import importlib
import utils as utils
utils = importlib.reload(utils)
from utils import iniezionetagimmagini, riassuntodocumento, metadata_extraction
import sqlite3

load_dotenv()

True

## SETTINGS GENERALI

In [ ]:
collectionname = "WAMASRAGBASE"

url_embedder = os.getenv("VLLM_API_BASE_URL")

url_qdrant = os.getenv("QDRANT_URL")

client = qdrant_client.QdrantClient(url=url_qdrant)

embed_model = OpenAIEmbedding(
    api_base=url_embedder,
    model_name="BAAI/bge-m3",
    api_key="null",
)

Settings.embed_model = embed_model

# Vector store
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collectionname,
    enable_hybrid=True,
    dense_vector_name="bge_m3",
    sparse_vector_name="bm25",
    fastembed_sparse_model="Qdrant/bm25"
)

storage_context = StorageContext.from_defaults(vector_store=vector_store)


2026-02-17 17:25:42,673 - INFO - HTTP Request: GET http://10.1.1.193:6333 "HTTP/1.1 200 OK"
2026-02-17 17:25:42,682 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASRAGBASEQWEN/exists "HTTP/1.1 200 OK"
2026-02-17 17:25:42,690 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASRAGBASEQWEN "HTTP/1.1 200 OK"
2026-02-17 17:25:42,692 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASRAGBASEQWEN/exists "HTTP/1.1 200 OK"
2026-02-17 17:25:42,695 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASRAGBASEQWEN "HTTP/1.1 200 OK"
2026-02-17 17:25:42,753 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASRAGBASEQWEN/exists "HTTP/1.1 200 OK"
2026-02-17 17:25:42,756 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASRAGBASEQWEN "HTTP/1.1 200 OK"


## Fare solo una volta!!! (o per testing)

In [ ]:
if client.collection_exists(collectionname):
    client.delete_collection(collectionname)

client.create_collection(
    collection_name=collectionname,
    vectors_config={
        "bge_m3": VectorParams(
            size=1024,   
            distance=Distance.COSINE
        )
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams()
    }
)

2026-02-17 17:25:54,158 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASRAGBASEQWEN/exists "HTTP/1.1 200 OK"


2026-02-17 17:25:54,174 - INFO - HTTP Request: DELETE http://10.1.1.193:6333/collections/WAMASRAGBASEQWEN "HTTP/1.1 200 OK"
2026-02-17 17:25:54,423 - INFO - HTTP Request: PUT http://10.1.1.193:6333/collections/WAMASRAGBASEQWEN "HTTP/1.1 200 OK"


True

In [18]:
if os.path.exists(f"../{collectionname}.db"):
    os.remove(f"../{collectionname}.db")

conn = sqlite3.connect(f"../{collectionname}.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS document_chunks (
        filename TEXT,
        chunk_index INTEGER,
        text_content TEXT,
        PRIMARY KEY (filename, chunk_index)
    )
""")
conn.commit()

## FUNZIONE PER EMBEDDARE CON TAGIMAGE, RIASSUNTO E CONTEXTUALIZE

In [19]:
def qdrantembedding(DOC_SOURCE, DOC_SOURCE_MD):
    #caricamento del documento json
    doc = DoclingDocument.load_from_json(DOC_SOURCE)
    #metto il tag immagine
    doc = iniezionetagimmagini(doc)
    #convertitore = DocumentConverter()
    #doc = convertitore.convert(DOC_SOURCE_MD)
    #doc = doc.document
    nodes = []

    sql_data = []


    DOC_SOURCE = DOC_SOURCE.split("/")[-1].replace(".json", ".pdf")

    

    chunker = HybridChunker()

    #doc_summary = riassuntodocumento(DOC_SOURCE_MD)
    chunks = list(chunker.chunk(doc))
    for i,chunk in enumerate(chunks):

        #enriched_text = doc_summary + "\n" + chunker.contextualize(chunk=chunk)
        enriched_text = chunker.contextualize(chunk=chunk) #OCCHIO AL DB CHE ORA CONTIENE SOLO IL TESTO DEL CHUNK SENZA IL RIASSUNTO, IL RIASSUNTO è IN UN'ALTRA COLONNA
        
        
        clean_meta = metadata_extraction(chunk)
        clean_meta["chunk_index"] = i

        sql_data.append((clean_meta.get("origin_filename", ""), i, enriched_text))

        new_node = TextNode(
            text=enriched_text,
            metadata=clean_meta
        )

        nodes.append(new_node)

    cursor.executemany(
        "INSERT OR REPLACE INTO document_chunks (filename, chunk_index, text_content) VALUES (?, ?, ?)", 
        sql_data
    )
    conn.commit()

    # INDICIZZAZIONE DEI NODI IN QDRANT

    index = VectorStoreIndex(
        nodes,
        storage_context=storage_context,
        show_progress=True
    )

    print(f"documento {DOC_SOURCE} embeddato")


Cuore dello script

In [20]:
base_folder = "../preprocessing/scratch"


json_file = ""
md_file = ""


for root, dirs, files in os.walk(base_folder):
    for file in files:
        if file.endswith(".json"):
            json_file = os.path.join(root, file)
        elif file.endswith(".md"):
            md_file = os.path.join(root, file)
    if json_file and md_file:
        #print(json_file)
        qdrantembedding(DOC_SOURCE=json_file, DOC_SOURCE_MD=md_file)



2026-02-17 17:26:13,452 - INFO - Migrating deprecated `annotations` to `meta`; this will be removed in the future. Note that only the first available instance of each annotation type will be migrated.
2026-02-17 17:26:13,452 - INFO - Migrating deprecated `annotations` to `meta`; this will be removed in the future. Note that only the first available instance of each annotation type will be migrated.
2026-02-17 17:26:13,453 - INFO - Migrating deprecated `annotations` to `meta`; this will be removed in the future. Note that only the first available instance of each annotation type will be migrated.
2026-02-17 17:26:13,453 - INFO - Migrating deprecated `annotations` to `meta`; this will be removed in the future. Note that only the first available instance of each annotation type will be migrated.
2026-02-17 17:26:13,453 - INFO - Migrating deprecated `annotations` to `meta`; this will be removed in the future. Note that only the first available instance of each annotation type will be migra

Generating embeddings:   0%|          | 0/16 [00:00<?, ?it/s]

2026-02-17 17:26:17,225 - INFO - HTTP Request: POST http://10.1.2.98/v1/embeddings "HTTP/1.1 502 Bad Gateway"
2026-02-17 17:26:17,227 - INFO - Retrying request to /embeddings in 0.448758 seconds
2026-02-17 17:26:20,745 - INFO - HTTP Request: POST http://10.1.2.98/v1/embeddings "HTTP/1.1 502 Bad Gateway"
2026-02-17 17:26:20,747 - INFO - Retrying request to /embeddings in 0.783639 seconds


KeyboardInterrupt: 